In [ ]:
!pip install -q transformers accelerate datasets evaluate rouge-score sentencepiece

In [ ]:
from datasets import load_dataset, Dataset

# WikiLingua: parquet-native Arabic article/summary pairs from WikiHow.
raw = load_dataset("esdurmus/wiki_lingua", "arabic", split="train")

pairs = []
for row in raw:
    docs = row["article"]["document"]
    sums = row["article"]["summary"]
    for d, s in zip(docs, sums):
        d, s = d.strip(), s.strip()
        if d and s:
            pairs.append({"text": d, "summary": s})

# Small subset — 1k train + 50 test is plenty for a short adaptation run.
flat = Dataset.from_list(pairs).shuffle(seed=42)
dataset = {
    "train": flat.select(range(1000)),
    "test":  flat.select(range(1000, 1050)),
}

print(f"Train: {len(dataset['train'])} samples")
print(f"Test:  {len(dataset['test'])} samples")
print(f"\n--- Sample ---")
print(f"Text:    {dataset['train'][0]['text'][:200]}...")
print(f"Summary: {dataset['train'][0]['summary'][:200]}...")

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# mT5 multilingual XLSum: already fine-tuned on XLSum (which includes Arabic).
# Strong Arabic baseline out of the box (ROUGE-1: 34.91 on Arabic XLSum).
# Standard seq2seq — no QLoRA / chat templates / token_type_ids needed.
MODEL_ID = "csebuetnlp/mT5_multilingual_XLSum"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_ID)

print(f"Model parameters: {model.num_parameters():,}")

In [ ]:
import re

WS = lambda k: re.sub(r"\s+", " ", re.sub(r"\n+", " ", k.strip()))

def preprocess(example):
    inputs = tokenizer(
        WS(example["text"]),
        max_length=512,
        truncation=True,
    )
    labels = tokenizer(
        WS(example["summary"]),
        max_length=128,
        truncation=True,
    )
    inputs["labels"] = labels["input_ids"]
    return inputs

train_dataset = dataset["train"].map(preprocess, remove_columns=["text", "summary"])
print(f"Tokenized train: {len(train_dataset)} samples")

In [ ]:
from transformers import (
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
)

OUTPUT_DIR = "./mt5-arabic-summarization"

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=5e-5,
    bf16=True,
    logging_steps=10,
    save_strategy="no",
    report_to="none",
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=data_collator,
    processing_class=tokenizer,
)

print("Starting training...")
trainer.train()

In [ ]:
import torch
from rouge_score import rouge_scorer
import numpy as np

test_samples = dataset["test"].select(range(20))

scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=False)
all_scores = {"rouge1": [], "rouge2": [], "rougeL": []}

for sample in test_samples:
    input_ids = tokenizer(
        WS(sample["text"]),
        return_tensors="pt",
        max_length=512,
        truncation=True,
    )["input_ids"].to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            input_ids=input_ids,
            max_length=128,
            num_beams=4,
            no_repeat_ngram_size=2,
        )[0]

    prediction = tokenizer.decode(output_ids, skip_special_tokens=True)
    scores = scorer.score(sample["summary"], prediction)
    for key in all_scores:
        all_scores[key].append(scores[key].fmeasure)

print("\nROUGE Evaluation Results")
print("=" * 40)
for key, values in all_scores.items():
    print(f"  {key}: {np.mean(values):.4f}")

In [ ]:
model.save_pretrained(f"{OUTPUT_DIR}/final")
tokenizer.save_pretrained(f"{OUTPUT_DIR}/final")
print(f"Model saved to {OUTPUT_DIR}/final")

from google.colab import drive
drive.mount("/content/drive")

!cp -r {OUTPUT_DIR}/final /content/drive/MyDrive/mt5-arabic-summarization
print("Model copied to Google Drive!")

In [ ]:
test_text = """أعلنت وزارة التعليم في المملكة العربية السعودية عن خطة جديدة لتطوير المناهج الدراسية
تتضمن إدخال مواد جديدة في مجال الذكاء الاصطناعي والبرمجة في المرحلة الثانوية.
وتهدف الخطة إلى تأهيل الطلاب لسوق العمل المستقبلي وتعزيز مهاراتهم الرقمية.
كما تشمل الخطة تدريب أكثر من عشرة آلاف معلم على التقنيات الحديثة خلال العام الدراسي القادم."""

input_ids = tokenizer(
    WS(test_text),
    return_tensors="pt",
    max_length=512,
    truncation=True,
)["input_ids"].to(model.device)

with torch.no_grad():
    output_ids = model.generate(
        input_ids=input_ids,
        max_length=128,
        num_beams=4,
        no_repeat_ngram_size=2,
    )[0]

summary = tokenizer.decode(output_ids, skip_special_tokens=True)
print(f"Input: {test_text[:100]}...\n")
print(f"Summary: {summary}")